In [ ]:
!pip install -q --no-deps boltz==2.2.1
!pip install -q "torch>=2.2" "numpy<2.0,>=1.26" hydra-core==1.3.2 pytorch-lightning==2.5.0 "rdkit>=2024.3.2" requests==2.32.3 "pandas>=2.2.2" types-requests einops==0.8.0 einx==0.3.0 fairscale==0.4.13 mashumaro==3.14 modelcif==1.2 wandb==0.18.7 click==8.1.7 pyyaml==6.0.2 biopython==1.84 scipy==1.13.1 numba==0.61.0 gemmi==0.6.5 scikit-learn==1.6.1 chembl_structure_pipeline==1.2.2
import torch, boltz
print("torch", torch.__version__, "cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("boltz", boltz.__file__)
try:
    import dm_tree
    print("dm_tree present (unexpected)")
except ImportError:
    print("dm_tree absent as intended; boltz imports fine without it")


In [ ]:
TARGET = 'KKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCKLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLRGC'
DESIGNS = {'r03_l07': 'SVTVDTYADPGDAPMVDHMAVTLGDHVEEAEVFTTENADITVRYKDKHGKVHEHTFTAPPEGTEYTVEIP', 'r03_l05': 'SVTVDTAAEPGDTPVVDHATVTLGDHVEEAEVTVAENGTITVRYKDKHGKVHEHTFTAPPEGTAYTVEIP', 'r01_l07': 'FADCESMFDFPANSAHTVEAPHGSKVIDSCLEEGPDSKTLKITFINEHGEVKETTLTVKKDESFCFD', 'r02_l07': 'DLYEQAEALAKEGKVEEALELLRKLIEEHPADLFHAILDLTIEPNGKKILEELAKEHPELKELIDEALLLN'}

import os
os.makedirs("/kaggle/working/yamls", exist_ok=True)
for name, bseq in DESIGNS.items():
    yaml = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {bseq}
      msa: empty
  - protein:
      id: B
      sequence: {TARGET}
      msa: empty
"""
    p = f"/kaggle/working/yamls/{name}.yaml"
    open(p, "w").write(yaml)
    print("wrote", p, f"binder={len(bseq)} target={len(TARGET)}")


In [ ]:
import subprocess, time, os
os.makedirs("/kaggle/working/preds", exist_ok=True)
for name in DESIGNS:
    y = f"/kaggle/working/yamls/{name}.yaml"
    t0 = time.time()
    print(f"=== folding {name} ===", flush=True)
    r = subprocess.run(
        ["boltz", "predict", y, "--out_dir", "/kaggle/working/preds",
         "--cache", "/kaggle/working/boltz_cache"],
        capture_output=True, text=True, timeout=7200)
    print(r.stdout[-3000:])
    print(r.stderr[-2000:])
    print(f"rc={r.returncode} time={time.time()-t0:.0f}s", flush=True)
print("ALL DONE")


In [ ]:
import json, glob, os
for name in DESIGNS:
    pat = f"/kaggle/working/preds/boltz_results_{name}/predictions_{name}/confidence_*_model_0.json"
    files = glob.glob(pat)
    print(name, "->", files)
    for f in files:
        c = json.load(open(f))
        keys = {k: (round(v,4) if isinstance(v,float) else v) for k,v in c.items()
                if k in ("iptm","ptm","ligand_iptm","protein_iptm","complex_plddt","complex_piplddt")}
        print("  ", keys)


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/boltz2_pathB_folds", "zip", "/kaggle/working/preds")
print("zipped -> /kaggle/working/boltz2_pathB_folds.zip")
